<a href="https://colab.research.google.com/github/Parmitha-566/movie-recommendation-system/blob/main/01_movie_recommendation_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Load the dataset directly from a URL
url = "https://raw.githubusercontent.com/YBI-Foundation/Dataset/main/Movies%20Recommendation.csv"
df = pd.read_csv(url)

# Take a look at the first 3 rows
df.head(3)

,Movie_ID,Movie_Title,Movie_Genre,Movie_Language,Movie_Budget,Movie_Popularity,Movie_Release_Date,Movie_Revenue,Movie_Runtime,Movie_Vote,...,Movie_Homepage,Movie_Keywords,Movie_Overview,Movie_Production_House,Movie_Production_Country,Movie_Spoken_Language,Movie_Tagline,Movie_Cast,Movie_Crew,Movie_Director
0,1,Four Rooms,Crime Comedy,en,4000000,22.876230,09-12-1995,4300000,98.0,6.5,...,NaN,hotel new year's eve witch bet hotel room,It's Ted the Bellhop's first night on the job....,"[{""name"": ""Miramax Films"", ""id"": 14}, {""name"":...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...","[{""iso_639_1"": ""en"", ""name"": ""English""}]",Twelve outrageous guests. Four scandalous requ...,Tim Roth Antonio Banderas Jennifer Beals Madon...,"[{'name': 'Allison Anders', 'gender': 1, 'depa...",Allison Anders
1,2,Star Wars,Adventure Action Science Fiction,en,11000000,126.393695,25-05-1977,775398007,121.0,8.1,...,http://www.starwars.com/films/star-wars-episod...,android galaxy hermit death star lightsaber,Princess Leia is captured and held hostage by ...,"[{""name"": ""Lucasfilm"", ""id"": 1}, {""name"": ""Twe...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...","[{""iso_639_1"": ""en"", ""name"": ""English""}]","A long time ago in a galaxy far, far away...",Mark Hamill Harrison Ford Carrie Fisher Peter ...,"[{'name': 'George Lucas', 'gender': 2, 'depart...",George Lucas
2,3,Finding Nemo,Animation Family,en,94000000,85.688789,30-05-2003,940335536,100.0,7.6,...,http://movies.disney.com/finding-nemo,father son relationship harbor underwater fish...,"Nemo, an adventurous young clownfish, is unexp...","[{""name"": ""Pixar Animation Studios"", ""id"": 3}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...","[{""iso_639_1"": ""en"", ""name"": ""English""}]","There are 3.7 trillion fish in the ocean, they...",Albert Brooks Ellen DeGeneres Alexander Gould ...,"[{'name': 'Andrew Stanton', 'gender': 2, 'depa...",Andrew Stanton


In [ ]:
# 1. Choose which columns describe the movie's "flavor"
selected_features = ['Movie_Genre', 'Movie_Keywords', 'Movie_Tagline', 'Movie_Cast', 'Movie_Director']

# 2. Fill missing values (NaN) with an empty string
for feature in selected_features:
    df[feature] = df[feature].fillna('')

# 3. Combine them into one single text column
df['combined_features'] = (
    df['Movie_Genre'] + ' ' +
    df['Movie_Keywords'] + ' ' +
    df['Movie_Tagline'] + ' ' +
    df['Movie_Cast'] + ' ' +
    df['Movie_Director']
)

# 4. Check what the combined soup looks like for the first movie
print(df['combined_features'].iloc[0][:200])

Crime Comedy hotel new year's eve witch bet hotel room Twelve outrageous guests. Four scandalous requests. And one lone bellhop, in his first day on the job, who's in for the wildest New year's Eve of


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# 1. Initialize the tool
cv = CountVectorizer(max_features=5000, stop_words='english')

# 2. Learn the vocabulary and convert text into a matrix of counts
feature_vectors = cv.fit_transform(df['combined_features'])

# 3. Print the shape of the resulting matrix
print(f"Matrix shape: {feature_vectors.shape}")

Matrix shape: (4760, 5000)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate the cosine similarity between every single movie pair
similarity = cosine_similarity(feature_vectors)

# Let's inspect the similarity matrix
print(f"Similarity matrix shape: {similarity.shape}")
print(f"First movie's similarity with itself: {similarity[0][0]:.2f}")

Similarity matrix shape: (4760, 4760)
First movie's similarity with itself: 1.00


In [ ]:
def recommend(movie_name):
    # 1. Look for the movie title (case-insensitive)
    matches = df[df['Movie_Title'].str.lower() == movie_name.lower()]

    if matches.empty:
        print(f"Movie '{movie_name}' not found. Check your spelling!")
        return

    # 2. Get the row index of the matched movie
    movie_index = matches.index[0]
    actual_title = df.iloc[movie_index]['Movie_Title']

    # 3. Get all similarity scores for this movie, keeping track of original indices
    # enumerate creates pairs like: [(0, 0.12), (1, 0.85), (2, 0.04), ...]
    movie_scores = list(enumerate(similarity[movie_index]))

    # 4. Sort the movies based on similarity score (item[1]) in descending order
    sorted_movies = sorted(movie_scores, key=lambda x: x[1], reverse=True)

    # 5. Take top 5 recommendations (skip index 0, which is the movie itself)
    top_5 = sorted_movies[1:6]

    print(f"\n==========================================")
    print(f"Top 5 Recommendations for: {actual_title}")
    print(f"==========================================")
    for rank, (index, score) in enumerate(top_5, 1):
        recommended_title = df.iloc[index]['Movie_Title']
        print(f"{rank}. {recommended_title} (Similarity: {score:.2f})")

In [ ]:
recommend("Avatar")
recommend("The Dark Knight Rises")
recommend("Toy Story")


Top 5 Recommendations for: Avatar
1. Guardians of the Galaxy (Similarity: 0.40)
2. Aliens (Similarity: 0.40)
3. Star Wars: Clone Wars: Volume 1 (Similarity: 0.37)
4. Alien (Similarity: 0.36)
5. Moonraker (Similarity: 0.33)

Top 5 Recommendations for: The Dark Knight Rises
1. The Dark Knight (Similarity: 0.68)
2. Batman Begins (Similarity: 0.68)
3. Amidst the Devil's Wings (Similarity: 0.44)
4. The Killer Inside Me (Similarity: 0.37)
5. The Prestige (Similarity: 0.36)

Top 5 Recommendations for: Toy Story
1. Toy Story 2 (Similarity: 0.54)
2. Toy Story 3 (Similarity: 0.46)
3. Cars 2 (Similarity: 0.30)
4. My Big Fat Greek Wedding 2 (Similarity: 0.29)
5. Animal House (Similarity: 0.29)
